 # Data cleaning

In [12]:
import pandas as pd

df = pd.read_csv("porto/porto/porto.csv")

rows_before = len(df)
deleted = {}
edited = {}
edited_rows = pd.Index([])

print(f"Loaded {rows_before:,} rows")

Loaded 1,710,670 rows


## DAY_TYPE
All trips have DAY_TYPE = A, so the column carries no information and is removed.

In [13]:
print(f"DAY_TYPE values: {df['DAY_TYPE'].value_counts().to_dict()}")
df = df.drop(columns="DAY_TYPE")

edited["DAY_TYPE column removed"] = len(df)
edited_rows = edited_rows.union(df.index)
print(f"Removed DAY_TYPE from {len(df):,} rows")
print(f"Rows in dataset: {len(df):,}")

DAY_TYPE values: {'A': 1710670}
Removed DAY_TYPE from 1,710,670 rows
Rows in dataset: 1,710,670


## Exact duplicate rows

In [14]:
before = len(df)
df = df.drop_duplicates()

deleted["Exact duplicate rows"] = before - len(df)
print(f"Deleted {deleted['Exact duplicate rows']:,} rows")
print(f"Rows in dataset: {len(df):,}")

Deleted 3 rows
Rows in dataset: 1,710,667


## Duplicate TRIP_IDs
Keep the row with the most GPS points for each TRIP_ID.

In [15]:
before = len(df)
n_points = df["POLYLINE"].str.count(r"\[") - 1
keep = n_points.groupby(df["TRIP_ID"]).idxmax()
df = df.loc[keep.sort_values()]

deleted["Duplicate TRIP_IDs"] = before - len(df)
print(f"Deleted {deleted['Duplicate TRIP_IDs']:,} rows")
print(f"Rows in dataset: {len(df):,}")

Deleted 78 rows
Rows in dataset: 1,710,589


## MISSING_DATA
Most trips with MISSING_DATA = True contain unrealistic jumps between GPS points (see EDA), so they are removed. After that every row has MISSING_DATA = False, so the column carries no information and is removed too.

In [16]:
before = len(df)
df = df[~df["MISSING_DATA"]]

deleted["MISSING_DATA = True"] = before - len(df)
print(f"Deleted {deleted['MISSING_DATA = True']:,} rows")

df = df.drop(columns="MISSING_DATA")
edited["MISSING_DATA column removed"] = len(df)
edited_rows = edited_rows.union(df.index)
print(f"Removed MISSING_DATA from {len(df):,} rows")
print(f"Rows in dataset: {len(df):,}")

Deleted 10 rows
Removed MISSING_DATA from 1,710,579 rows
Rows in dataset: 1,710,579


## ORIGIN_CALL / ORIGIN_STAND
Convert from float (caused by NaN) to nullable integers. B trips without ORIGIN_STAND are kept with NULL.

In [17]:
has_value = df["ORIGIN_CALL"].notna() | df["ORIGIN_STAND"].notna()
df["ORIGIN_CALL"] = df["ORIGIN_CALL"].astype("Int64")
df["ORIGIN_STAND"] = df["ORIGIN_STAND"].astype("Int64")

edited["ORIGIN_CALL / ORIGIN_STAND"] = has_value.sum()
edited_rows = edited_rows.union(df.index[has_value])
b_without_stand = ((df["CALL_TYPE"] == "B") & df["ORIGIN_STAND"].isna()).sum()
print(f"Edited {edited['ORIGIN_CALL / ORIGIN_STAND']:,} rows (float -> int)")
print(f"Kept {b_without_stand:,} B trips with ORIGIN_STAND = NULL")
print(f"Rows in dataset: {len(df):,}")

Edited 1,171,325 rows (float -> int)
Kept 11,302 B trips with ORIGIN_STAND = NULL
Rows in dataset: 1,710,579


## TIMESTAMP
Convert Unix time to DATETIME in Porto local time.

In [18]:
df["TIMESTAMP"] = (
    pd.to_datetime(df["TIMESTAMP"], unit="s", utc=True)
    .dt.tz_convert("Europe/Lisbon")
    .dt.tz_localize(None)
)

edited["TIMESTAMP"] = len(df)
edited_rows = edited_rows.union(df.index)
print(f"Edited {edited['TIMESTAMP']:,} rows (Unix time -> datetime)")
print(f"Rows in dataset: {len(df):,}")

Edited 1,710,579 rows (Unix time -> datetime)
Rows in dataset: 1,710,579


## Check TRIP_ID
Must be a unique 64-bit integer to be used as BIGINT primary key.

In [19]:
was_int64 = df["TRIP_ID"].dtype == "int64"
df["TRIP_ID"] = df["TRIP_ID"].astype("int64")
assert df["TRIP_ID"].is_unique

edited["TRIP_ID"] = 0 if was_int64 else len(df)
if not was_int64:
    edited_rows = edited_rows.union(df.index)
print(f"Edited {edited['TRIP_ID']:,} rows, all TRIP_IDs unique")
print(f"Rows in dataset: {len(df):,}")

Edited 0 rows, all TRIP_IDs unique
Rows in dataset: 1,710,579


## Summary

In [20]:
print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {len(df):,}")

print("\nDeleted")
for step, n in deleted.items():
    print(f"  {step:<28} {n:>10,}")
print("\nEdited")
for step, n in edited.items():
    print(f"  {step:<28} {n:>10,}")

print(f"\nTotal deleted rows: {rows_before - len(df):,}")
print(f"Total edited rows:  {len(edited_rows.intersection(df.index)):,} (each row counted once)")

Rows before: 1,710,670
Rows after:  1,710,579

Deleted
  Exact duplicate rows                  3
  Duplicate TRIP_IDs                   78
  MISSING_DATA = True                  10

Edited
  DAY_TYPE column removed       1,710,670
  MISSING_DATA column removed   1,710,579
  ORIGIN_CALL / ORIGIN_STAND    1,171,325
  TIMESTAMP                     1,710,579
  TRIP_ID                               0

Total deleted rows: 91
Total edited rows:  1,710,579 (each row counted once)


## Overview

In [21]:
print(f"{len(df):,} rows, {df.shape[1]} columns")
width = max(map(len, df.columns))
for col, dtype in df.dtypes.items():
    print(f"  {col:<{width}}  {dtype}")
print(f"First trip: {df['TIMESTAMP'].min():%Y-%m-%d %H:%M} (Porto time)")
print(f"Last trip:  {df['TIMESTAMP'].max():%Y-%m-%d %H:%M} (Porto time)")

1,710,579 rows, 7 columns
  TRIP_ID       int64
  CALL_TYPE     str
  ORIGIN_CALL   Int64
  ORIGIN_STAND  Int64
  TAXI_ID       int64
  TIMESTAMP     datetime64[s]
  POLYLINE      str
First trip: 2013-07-01 01:00 (Porto time)
Last trip:  2014-07-01 00:59 (Porto time)


## Export

In [22]:
# Parquet keeps the Int64 / datetime / bool dtypes; main.py reads this file and seeds MySQL
CLEAN_DATA_PATH = "porto/clean.parquet"
df.to_parquet(CLEAN_DATA_PATH, index=False)
print(f"Wrote {len(df):,} rows, {df.shape[1]} columns to {CLEAN_DATA_PATH}")

Wrote 1,710,579 rows, 7 columns to porto/clean.parquet
